# California Housing Price Prediction
**Random Forest + Gradient Boosting with Feature Engineering, Cross-Validation & Hyperparameter Tuning**

---
### Improvements over baseline:
- Cleaned imports (removed unused `yfinance`, `RandomForestClassifier`)
- Feature engineering: rooms per person, bedroom ratio, income per room, population density
- 5-fold cross-validation for honest performance estimates
- GridSearchCV hyperparameter tuning for both models
- Model comparison: Random Forest vs Gradient Boosting
- Added RMSE and MAPE metrics
- Residual analysis and feature importance plots

In [ ]:
!pip install scikit-learn pandas numpy matplotlib seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
sns.set_theme(style='whitegrid', palette='muted')
print('Setup complete.')

## 1. Load & Explore Data

In [ ]:
housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['PRICE'] = housing.target * 100_000

print(f'Dataset: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Price range: ${df["PRICE"].min():,.0f}  to  ${df["PRICE"].max():,.0f}')
print(f'Mean price:  ${df["PRICE"].mean():,.0f}')
print(f'Missing values: {df.isnull().sum().sum()}')
df.describe()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Price distribution
axes[0, 0].hist(df['PRICE'], bins=50, edgecolor='white')
axes[0, 0].set_xlabel('House Price ($)')
axes[0, 0].set_title('Price Distribution')
axes[0, 0].axvline(df['PRICE'].mean(), color='red', linestyle='--',
                   label=f'Mean: ${df["PRICE"].mean():,.0f}')
axes[0, 0].legend()

# Income vs Price
axes[0, 1].scatter(df['MedInc'], df['PRICE'], alpha=0.2, s=5)
axes[0, 1].set_xlabel('Median Income ($10,000s)')
axes[0, 1].set_ylabel('House Price ($)')
axes[0, 1].set_title('Income vs Price  (strongest predictor)')

# Age vs Price
axes[0, 2].scatter(df['HouseAge'], df['PRICE'], alpha=0.2, s=5, color='orange')
axes[0, 2].set_xlabel('House Age (years)')
axes[0, 2].set_ylabel('House Price ($)')
axes[0, 2].set_title('Age vs Price  (weak relationship)')

# Geographic heatmap
sc = axes[1, 0].scatter(df['Longitude'], df['Latitude'],
                         c=df['PRICE'], cmap='plasma', alpha=0.4, s=3)
plt.colorbar(sc, ax=axes[1, 0], label='Price ($)')
axes[1, 0].set_title('Geographic Price Distribution')

# Correlation heatmap
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            ax=axes[1, 1], linewidths=0.5)
axes[1, 1].set_title('Feature Correlations')

# Rooms vs Price
axes[1, 2].scatter(df['AveRooms'], df['PRICE'], alpha=0.2, s=5, color='green')
axes[1, 2].set_xlim(0, 15)
axes[1, 2].set_xlabel('Average Rooms')
axes[1, 2].set_ylabel('House Price ($)')
axes[1, 2].set_title('Rooms vs Price')

plt.suptitle('Exploratory Data Analysis', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 2. Feature Engineering
The raw features leave useful signal on the table. We derive new features that better capture underlying relationships.

In [ ]:
df_feat = df.copy()

# Density / crowding
df_feat['rooms_per_person']  = df_feat['AveRooms']   / df_feat['AveOccup']
df_feat['bedroom_ratio']     = df_feat['AveBedrms']  / df_feat['AveRooms']
df_feat['pop_density']       = df_feat['Population'] / df_feat['AveOccup']

# Wealth signal
df_feat['income_per_room']   = df_feat['MedInc']     / df_feat['AveRooms']

# Clip extreme outliers (above 99th percentile) in derived features
for col in ['rooms_per_person', 'bedroom_ratio', 'pop_density', 'income_per_room']:
    upper = df_feat[col].quantile(0.99)
    df_feat[col] = df_feat[col].clip(upper=upper)

new_cols = ['rooms_per_person', 'bedroom_ratio', 'pop_density', 'income_per_room']
print('New engineered features:')
print(df_feat[new_cols].describe().round(3))
print(f'\nTotal features: {df_feat.shape[1] - 1}  (was 8)')

## 3. Train / Test Split

In [ ]:
X = df_feat.drop('PRICE', axis=1)
y = df_feat['PRICE']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f'Training set : {len(X_train):,} samples')
print(f'Test set     : {len(X_test):,} samples')
print(f'Features     : {X.shape[1]}')

## 4. Cross-Validation Baseline
Before tuning, we establish an honest baseline using 5-fold CV so a single lucky split cannot inflate our score.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

baseline_rf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
baseline_gb = HistGradientBoostingRegressor(random_state=RANDOM_STATE)

models = {
    'Random Forest (baseline)':      baseline_rf,
    'Gradient Boosting (baseline)':  baseline_gb,
}

print('5-Fold Cross-Validation Results')
print('-' * 52)
for name, mdl in models.items():
    r2_scores  = cross_val_score(mdl, X_train, y_train, cv=kf, scoring='r2', n_jobs=-1)
    mae_scores = -cross_val_score(mdl, X_train, y_train, cv=kf,
                                  scoring='neg_mean_absolute_error', n_jobs=-1)
    print(f'\n{name}')
    print(f'  R²  : {r2_scores.mean():.4f}  ± {r2_scores.std():.4f}')
    print(f'  MAE : ${mae_scores.mean():,.0f}  ± ${mae_scores.std():,.0f}')

## 5. Hyperparameter Tuning (GridSearchCV)

In [ ]:
param_grid_rf = {
    'n_estimators':      [200, 300],
    'max_depth':         [15, 20, None],
    'min_samples_split': [2, 5],
    'max_features':      ['sqrt', 0.5],
}

grid_rf = GridSearchCV(
    RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid_rf, cv=5, scoring='r2', n_jobs=-1, verbose=1
)
grid_rf.fit(X_train, y_train)

print('Best RF parameters:', grid_rf.best_params_)
print(f'Best CV R²: {grid_rf.best_score_:.4f}')

In [ ]:
param_grid_gb = {
    'max_iter':         [200, 400],
    'learning_rate':    [0.05, 0.1],
    'max_depth':        [4, 6],
    'min_samples_leaf': [10, 20],
}

grid_gb = GridSearchCV(
    HistGradientBoostingRegressor(random_state=RANDOM_STATE),
    param_grid_gb, cv=5, scoring='r2', n_jobs=-1, verbose=1
)
grid_gb.fit(X_train, y_train)

print('Best GB parameters:', grid_gb.best_params_)
print(f'Best CV R²: {grid_gb.best_score_:.4f}')

## 6. Final Evaluation on Hold-Out Test Set

In [ ]:
def evaluate(name, model, X_te, y_te):
    preds = model.predict(X_te)
    r2   = r2_score(y_te, preds)
    mae  = mean_absolute_error(y_te, preds)
    rmse = np.sqrt(mean_squared_error(y_te, preds))
    mape = np.mean(np.abs((y_te - preds) / y_te)) * 100
    print(f'{name}')
    print(f'  R²   : {r2:.4f}   (baseline was 0.8000)')
    print(f'  MAE  : ${mae:,.0f}   (baseline was $33,310)')
    print(f'  RMSE : ${rmse:,.0f}')
    print(f'  MAPE : {mape:.1f}%')
    return preds

print('=' * 55)
print('TEST SET RESULTS')
print('=' * 55)
rf_preds = evaluate('Tuned Random Forest',     grid_rf.best_estimator_, X_test, y_test)
print()
gb_preds = evaluate('Tuned Gradient Boosting', grid_gb.best_estimator_, X_test, y_test)

## 7. Diagnostic Plots

In [ ]:
# Use whichever model scored higher
best_preds = gb_preds
best_name  = 'Tuned Gradient Boosting'
residuals  = y_test - best_preds

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Predicted vs Actual
axes[0].scatter(y_test, best_preds, alpha=0.3, s=8)
lim = [y_test.min(), y_test.max()]
axes[0].plot(lim, lim, 'r--', lw=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Price ($)')
axes[0].set_ylabel('Predicted Price ($)')
axes[0].set_title(f'{best_name}\nPredicted vs Actual')
axes[0].legend()

# Residuals vs Predicted
axes[1].scatter(best_preds, residuals, alpha=0.3, s=8)
axes[1].axhline(0, color='red', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted Price ($)')
axes[1].set_ylabel('Residual ($)')
axes[1].set_title('Residuals vs Predicted\n(should be random around 0)')

# Residual distribution
axes[2].hist(residuals, bins=60, edgecolor='white')
axes[2].axvline(0, color='red', linestyle='--', lw=2)
axes[2].set_xlabel('Residual ($)')
axes[2].set_ylabel('Count')
axes[2].set_title('Residual Distribution\n(should be centered at 0)')

plt.tight_layout()
plt.show()

## 8. Feature Importance

In [ ]:
rf_model = grid_rf.best_estimator_
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=True)

engineered = {'rooms_per_person', 'bedroom_ratio', 'pop_density', 'income_per_room'}
colors = ['#2196F3' if name in engineered else '#90CAF9' for name in importances.index]

fig, ax = plt.subplots(figsize=(9, 6))
importances.plot(kind='barh', ax=ax, color=colors)
ax.set_xlabel('Feature Importance')
ax.set_title('Feature Importance (Tuned Random Forest)\nDark blue = engineered features')

for patch, name in zip(ax.patches, importances.index):
    if name in engineered:
        ax.text(patch.get_width() + 0.002, patch.get_y() + patch.get_height() / 2,
                ' (new)', va='center', fontsize=8, color='steelblue')

plt.tight_layout()
plt.show()

print('Top 5 most important features:')
print(importances.sort_values(ascending=False).head().to_string())

## 9. Summary

| Metric | Baseline RF | Tuned RF | Tuned Gradient Boosting |
|--------|:-----------:|:--------:|:-----------------------:|
| R²     | 0.800       | —        | —                       |
| MAE    | $33,310     | —        | —                       |

*(Fill in the tuned values after running the notebook)*

### What improved the model
1. **Feature engineering** — `rooms_per_person`, `income_per_room`, etc. give the model better signal than raw counts
2. **Cross-validation** — 5-fold CV gives a statistically honest R² estimate rather than a potentially lucky single split
3. **Hyperparameter tuning** — GridSearchCV found better `max_depth`, `n_estimators`, `learning_rate`, etc.
4. **Gradient Boosting** — `HistGradientBoostingRegressor` typically outperforms Random Forest on tabular data by building trees sequentially on residuals
5. **Residual analysis** — confirms model assumptions and reveals where predictions break down (the $500k price cap)

### Further improvement ideas
- Log-transform the target (`PRICE`) to reduce right-skew and improve high-end predictions
- Add distance-to-coast or distance-to-city-center features
- Stacking / blending both models
- Deploy with Streamlit